In [ ]:
from astropy.io import fits
import os
import numpy as np
import pandas as pd
from itertools import zip_longest
import matplotlib.pyplot as plt
import random
import ipywidgets as widgets
from IPython.display import display


In [ ]:
# Path to data and grid selection:
path = '/data-fast/dragon/vicente/iongrid/runs/run_20261029_htc_cartesian/results'


In [ ]:
# Files:
line_emission_file = 'linem_clean.fits'
transmission_file = 'optdepth_clean.fits'

# Load data:
files = [line_emission_file, transmission_file]
spectra = []
param_names = []
param_values = []
param_units = []

for k, file_name in enumerate(files):
    with fits.open(os.path.join(path, file_name)) as hdul:
        if 'PARAMS' in hdul:
            # Read parameter rows and the shared energy axis.
            param_names.append(np.array(hdul['PARAMS'].columns.names))
            param_values.append(pd.DataFrame(hdul['PARAMS'].data))
            param_units.append(np.array(hdul['PARAMS'].columns.units))
            energy = hdul['ENERGY'].data
            
            if file_name == line_emission_file:
                spectra.append(hdul['SPECTRA'].data)
            elif file_name == transmission_file:
                spectra.append(np.exp(-hdul['SPECTRA'].data))
                
        elif 'GRID' in hdul:
            grid_hdu = hdul['GRID']
            cols = grid_hdu.columns
            
            # First column is ENERGY
            energy = grid_hdu.data[cols.names[0]][0]
            
            # Remaining columns are parameters
            p_names = cols.names[1:]
            p_units = cols.units[1:]
            
            param_names.append(np.array(p_names))
            param_units.append(np.array(p_units))
            
            # Extract parameter coordinate axes and create a full mesh/flat table
            p_axes = [grid_hdu.data[name][0] for name in p_names]
            mesh = np.meshgrid(*p_axes, indexing='ij')
            flat_vals = {name: m.ravel() for name, m in zip(p_names, mesh)}
            param_values.append(pd.DataFrame(flat_vals))
            
            # Load and flatten spectra to match the flat parameter rows: (total_models, n_energy)
            raw_spectra = hdul['SPECTRA'].data
            if file_name == transmission_file:
                raw_spectra = np.exp(-raw_spectra)
                
            n_energy = energy.shape[0]
            total_models = np.prod([len(ax) for ax in p_axes])
            spectra.append(raw_spectra.reshape(total_models, n_energy))

In [ ]:
# Take a first look at the parameter distribution:
components = [
    ('Emission', param_names[0], param_values[0], param_units[0]),
    ('Transmission', param_names[1], param_values[1], param_units[1])
]
max_cols = max(len(param_names[0]), len(param_names[1]))
fig, axs = plt.subplots(nrows=2, ncols=max_cols, figsize=(4 * max_cols, 8), squeeze=False)
nbins = 30
for row_idx, (title, names, values, units) in enumerate(components):
    for k in range(max_cols):
        if k < len(names):
            col_name = names[k]
            if col_name == 'POINT_ID':
                axs[row_idx, k].axis('off')
                continue
            unit = units[k]
            axs[row_idx, k].hist(values[col_name], bins=nbins)
            # Add parameter name and FITS unit to x-axis:
            if unit is not None and str(unit).strip():
                xlabel = f'{col_name} ({unit})'
            else:
                xlabel = col_name
            axs[row_idx, k].set_xlabel(xlabel)
            if k == 0:
                axs[row_idx, k].set_ylabel(f'{title}\nNumber of spectra')
        else:
            axs[row_idx, k].axis('off') 
plt.tight_layout()
plt.show()

In [ ]:
# Take a first look at the energy array:
plt.plot(np.linspace(0, len(energy) - 1, len(energy)), energy)
plt.xlabel('Channel')
plt.ylabel('Energy (keV)')
plt.yscale('log')
plt.show()

In [ ]:
# Take a first look at some random spectra:
number_spectra = 20

# Check if 'POINT_ID' exists in the parameter tables (typical for LHS, not Cartesian):
has_point_id = ('POINT_ID' in param_values[0].columns) and ('POINT_ID' in param_values[1].columns)
if has_point_id:
    # Find common POINT_IDs between emission and absorption components:
    common_ids = set(param_values[0]['POINT_ID']).intersection(set(param_values[1]['POINT_ID']))
    sampled_ids = random.sample(sorted(list(common_ids)), min(number_spectra, len(common_ids)))
else:
    # Fallback for Cartesian grids: sample random row indices directly
    max_idx = min(len(param_values[0]), len(param_values[1]))
    sampled_ids = range(min(number_spectra, max_idx))
labels = ['Emission lines', 'Transmission']
# Parameters to exclude from the panel titles:
excluded_params = ['point_id', 'population']

for gid in sampled_ids:
    if has_point_id:
        # Get all matching indices for this POINT_ID:
        matching_em = param_values[0].index[param_values[0]['POINT_ID'] == gid]
        matching_trans = param_values[1].index[param_values[1]['POINT_ID'] == gid]
        idx_em = random.choice(matching_em)
        idx_trans = random.choice(matching_trans)
    else:
        # For Cartesian, gid acts directly as the index
        idx_em = gid
        idx_trans = gid
    indices = [idx_em, idx_trans]
    # Figure:
    fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(14, 5))
    for j in range(2):
        k = indices[j]
        # Build parameter string, including FITS units:
        param_items = []
        for name, val, unit in zip(param_names[j], param_values[j].iloc[k], param_units[j]):
            if name.lower() in excluded_params:
                continue
            # Format numbers cleanly:
            val_str = (f"{val:.3g}" if isinstance(val, (float, np.floating)) else f"{val}")
            # Append parameter name, value, and unit:
            if unit is not None and str(unit).strip():
                param_items.append(f"{name} = {val_str} (${unit}$)")
            else:
                param_items.append(f"{name} = {val_str}")
        param_str = " ; ".join(param_items)
    
        # Plot spectrum:
        axs[j].plot(energy, spectra[j][k])
        axs[j].set_xscale('log')
        axs[j].set_xlabel('Energy (keV)')
        # Component name:
        axs[j].set_title(labels[j], fontsize=10, pad=25)
        # Parameter information below the component name:
        axs[j].text(0.5, 1.01, param_str, transform=axs[j].transAxes, fontsize=6, ha='center', va='bottom', wrap=True)
        if j == 0:
            axs[j].set_ylabel(r'Flux ($\mathrm{ph\ s^{-1}\ cm^{-2}}$)')
            axs[j].set_yscale('log')
        else:
            axs[j].set_ylabel('Transmitted fraction')
            
    fig.suptitle(f'grid_id: {gid}', fontsize=10)
    plt.tight_layout(rect=[0, 0, 1, 1])
    plt.show()

In [ ]:
# Pre-process parameter values:
for comp_idx in range(len(param_names)):
    df = param_values[comp_idx]
    names_list = list(param_names[comp_idx])
    rename_dict = {}
    for k, original_name in enumerate(names_list):
        name_lower = original_name.lower()
        if name_lower == "luminosity":
            df[original_name] = np.log10(df[original_name])
            rename_dict[original_name] = "logLum"
            names_list[k] = "logLum"
        elif name_lower == "vturb":
            df[original_name] = np.log10(df[original_name])
            rename_dict[original_name] = "logvturb"
            names_list[k] = "logvturb"
        elif name_lower == "density":
            df[original_name] = np.log10(df[original_name])
            rename_dict[original_name] = "logn"
            names_list[k] = "logn"
        elif name_lower == "gamma":
            df[original_name] = np.round(df[original_name], 3)
    # Update DataFrame column names to match the new parameter names
    df.rename(columns=rename_dict, inplace=True)
    param_names[comp_idx] = np.array(names_list)
# Look at the new distributions of parameters:
components = [
    ('Emission', param_names[0], param_values[0], param_units[0]),
    ('Transmission', param_names[1], param_values[1], param_units[1])
]
max_cols = max(len(param_names[0]), len(param_names[1]))
fig, axs = plt.subplots(nrows=2, ncols=max_cols, figsize=(4 * max_cols, 8), squeeze=False)
nbins = 30
for row_idx, (title, names, values, units) in enumerate(components):
    for k in range(max_cols):
        if k < len(names):
            col_name = names[k]
            if col_name == 'POINT_ID':
                axs[row_idx, k].axis('off')
                continue
            unit = units[k]
            axs[row_idx, k].hist(values[col_name], bins=nbins)
            # Add parameter name and FITS unit to x-axis:
            if unit is not None and str(unit).strip():
                xlabel = f'{col_name} ({unit})'
            else:
                xlabel = col_name
            axs[row_idx, k].set_xlabel(xlabel)
            if k == 0:
                axs[row_idx, k].set_ylabel(f'{title}\nNumber of spectra')
        else:
            axs[row_idx, k].axis('off') 
plt.tight_layout()
plt.show()

In [ ]:
# Energy selection:
already_selected = False
min_energy_keV = 2.0
max_energy_keV = 12.0

if already_selected:
    energy_range = energy.copy()
    spectra_range = [spec.copy() for spec in spectra]
else:
    # Mask energy bins within the selected bandpass:
    energy_limits = (energy >= min_energy_keV) & (energy <= max_energy_keV)
    if not np.any(energy_limits):
        raise ValueError(f"No energy channels found between {min_energy_keV} and {max_energy_keV} keV.")
    energy_range = energy[energy_limits]
    # Slice the energy dimension (axis 1) for both emission and absorption grids
    spectra_range = [spec[:, energy_limits] for spec in spectra]
print(f"Selected energy range: {energy_range[0]:.2f} - {energy_range[-1]:.2f} keV ({len(energy_range)} bins)")

In [ ]:
# Take a first look at some random spectra in the selected energy range:
number_spectra = 20

# Check if 'POINT_ID' exists in the parameter tables (typical for LHS, not Cartesian):
has_point_id = ('POINT_ID' in param_values[0].columns) and ('POINT_ID' in param_values[1].columns)
if has_point_id:
    # Find common POINT_IDs between emission and absorption components:
    common_ids = set(param_values[0]['POINT_ID']).intersection(set(param_values[1]['POINT_ID']))
    sampled_ids = random.sample(sorted(list(common_ids)), min(number_spectra, len(common_ids)))
else:
    # Fallback for Cartesian grids: sample random row indices directly
    max_idx = min(len(param_values[0]), len(param_values[1]))
    sampled_ids = range(min(number_spectra, max_idx))
labels = ['Emission lines', 'Transmission']
# Parameters to exclude from the panel titles:
excluded_params = ['point_id', 'population']

for gid in sampled_ids:
    if has_point_id:
        # Get all matching indices for this POINT_ID:
        matching_em = param_values[0].index[param_values[0]['POINT_ID'] == gid]
        matching_trans = param_values[1].index[param_values[1]['POINT_ID'] == gid]
        idx_em = random.choice(matching_em)
        idx_trans = random.choice(matching_trans)
    else:
        # For Cartesian, gid acts directly as the index
        idx_em = gid
        idx_trans = gid
    indices = [idx_em, idx_trans]
    # Figure:
    fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(14, 5))
    for j in range(2):
        k = indices[j]
        # Build parameter string, including FITS units:
        param_items = []
        for name, val, unit in zip(param_names[j], param_values[j].iloc[k], param_units[j]):
            if name.lower() in excluded_params:
                continue
            # Format numbers cleanly:
            val_str = (f"{val:.3g}" if isinstance(val, (float, np.floating)) else f"{val}")
            # Append parameter name, value, and unit:
            if unit is not None and str(unit).strip():
                param_items.append(f"{name} = {val_str} (${unit}$)")
            else:
                param_items.append(f"{name} = {val_str}")
        param_str = " ; ".join(param_items)
    
        # Plot spectrum:
        axs[j].plot(energy_range, spectra_range[j][k])
        axs[j].set_xscale('linear')
        axs[j].set_xlabel('Energy (keV)')
        # Component name:
        axs[j].set_title(labels[j], fontsize=10, pad=25)
        # Parameter information below the component name:
        axs[j].text(0.5, 1.01, param_str, transform=axs[j].transAxes, fontsize=6, ha='center', va='bottom', wrap=True)
        if j == 0:
            axs[j].set_ylabel(r'Flux ($\mathrm{ph\ s^{-1}\ cm^{-2}}$)')
            axs[j].set_yscale('log')
        else:
            axs[j].set_ylabel('Transmitted fraction')
            
    fig.suptitle(f'grid_id: {gid}', fontsize=10)
    plt.tight_layout(rect=[0, 0, 1, 1])
    plt.show()

In [ ]:
# Interactive one-parameter exploration:
labels = ['Emission lines', 'Transmission']
# Metadata that should not be considered physical parameters:
excluded_params = {'point_id', 'grid_id', 'population'}
# Check if 'POINT_ID' exists (LHS vs Cartesian):
has_point_id = ('POINT_ID' in param_values[0].columns) and ('POINT_ID' in param_values[1].columns)

# Physical parameters available in both components:
available_params = []
for names in param_names:
    for name in names:
        if name.lower() not in excluded_params and name not in available_params:
            available_params.append(name)

# Available reference IDs (POINT_IDs for LHS, row indices for Cartesian).
if has_point_id:
    common_ids = sorted(set(param_values[0]['POINT_ID']).intersection(set(param_values[1]['POINT_ID'])))
    ref_desc = 'Reference POINT_ID:'
else:
    max_models = min(len(param_values[0]), len(param_values[1]))
    common_ids = list(range(max_models))
    ref_desc = 'Reference Model Index:'

# Helper functions:
def values_match(series, value):
    """Compare numerical parameters robustly."""
    if pd.api.types.is_numeric_dtype(series):
        return np.isclose(
            series.to_numpy(dtype=float),
            float(value),
            rtol=1e-8,
            atol=1e-10,
            equal_nan=False
        )
    return (series == value).to_numpy()

def format_value(value):
    if isinstance(value, (float, np.floating)):
        return f'{value:.3g}'
    return str(value)

# Plotting function:
def explore_one_parameter(parameter, reference_id, plot_mode):
    fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(15, 5.5))
    fig.suptitle(f'Exploring parameter: {parameter} | {ref_desc} {reference_id}', fontsize=12)
    for j in range(2):
        df = param_values[j]
        # Get the reference row for this component:
        if has_point_id:
            ref_matches = df.index[df['POINT_ID'] == reference_id]
            ref_idx = ref_matches[0] if len(ref_matches) > 0 else 0
        else:
            ref_idx = int(reference_id)
        reference = df.loc[ref_idx]
        # Find the corresponding parameter in this component:
        matched_param = next((p for p in param_names[j] if p.lower() == parameter.lower()), None)
        if matched_param is None:
            axs[j].text(0.5, 0.5, 'Parameter not available', ha='center', va='center', transform=axs[j].transAxes)
            continue
        # Build parameter string for the frozen/non-explored parameters:
        param_items = []
        for name, val, unit in zip(param_names[j], reference, param_units[j]):
            if name.lower() in excluded_params:
                continue
            if name.lower() == parameter.lower():
                continue  # Skip the explored parameter since it varies along the curves
            val_str = format_value(val)
            if unit is not None and str(unit).strip():
                param_items.append(f"{name} = {val_str} (${unit}$)")
            else:
                param_items.append(f"{name} = {val_str}")
        param_str = " ; ".join(param_items)

        # Freeze all other physical parameters at their reference values:
        mask = np.ones(len(df), dtype=bool)
        for name in param_names[j]:
            if name.lower() in excluded_params:
                continue
            if name.lower() == parameter.lower():
                continue
            if name not in reference.index:
                continue
            mask &= values_match(df[name], reference[name])

        # Extract matching rows:
        matching_positions = np.flatnonzero(mask)
        if len(matching_positions) == 0:
            axs[j].text(0.5, 0.5, 'No matching spectra found', ha='center', va='center', transform=axs[j].transAxes)
            continue

        # Sort by the parameter being explored:
        matching_positions = sorted(matching_positions, key=lambda i: df.iloc[i][matched_param])

        # Keep only one spectrum per parameter value:
        unique_positions = []
        seen_values = []
        for pos in matching_positions:
            value = df.iloc[pos][matched_param]
            if not any(
                np.isclose(value, previous, rtol=1e-8, atol=1e-10)
                if isinstance(value, (int, float, np.number))
                and isinstance(previous, (int, float, np.number))
                else value == previous
                for previous in seen_values
            ):
                unique_positions.append(pos)
                seen_values.append(value)

        # Select all values or only extremes:
        if plot_mode == 'extremes' and len(unique_positions) > 2:
            middle = len(unique_positions) // 2
            unique_positions = [unique_positions[0], unique_positions[middle], unique_positions[-1]]
        # Plot all selected spectra.
        for pos in unique_positions:
            row = df.iloc[pos]
            value = row[matched_param]
            axs[j].plot(energy_range, spectra_range[j][pos], alpha=0.8, label=f'{matched_param} = {format_value(value)}')
        axs[j].set_title(labels[j], fontsize=11, pad=25)
        axs[j].text(0.5, 1.01, param_str, transform=axs[j].transAxes, fontsize=6, ha='center', va='bottom', wrap=True)
        axs[j].set_xlabel('Energy (keV)')
        axs[j].legend(fontsize=8)
        if j == 0:
            axs[j].set_yscale('log')
            axs[j].set_ylabel(r'Flux ($\mathrm{ph\ s^{-1}\ cm^{-2}}$)')
        else:
            axs[j].set_ylabel('Transmitted fraction')
    plt.tight_layout(rect=[0, 0, 1, 1])
    plt.show()

# Interactive controls:
parameter_widget = widgets.Dropdown(
    options=available_params,
    value='gamma' if 'gamma' in available_params else available_params[0],
    description='Parameter:',
    style={'description_width': 'initial'},
    layout={'width': '350px'}
)

reference_widget = widgets.Dropdown(
    options=common_ids,
    value=common_ids[0],
    description=ref_desc,
    style={'description_width': 'initial'},
    layout={'width': '350px'}
)

mode_widget = widgets.ToggleButtons(
    options=[('All values', 'all'), ('Minimum / Middle / Maximum', 'extremes')],
    value='all',
    description='Plot:',
    style={'description_width': 'initial'}
)

controls = widgets.VBox([
    parameter_widget,
    reference_widget,
    mode_widget
])

output = widgets.interactive_output(
    explore_one_parameter,
    {
        'parameter': parameter_widget,
        'reference_id': reference_widget,
        'plot_mode': mode_widget
    }
)
display(controls, output)


In [ ]:
# Configuration:
labels = ['Emission lines', 'Transmission']
excluded_params = {
    'point_id',
    'grid_id',
    'population'
}
# Check if POINT_ID exists (LHS vs Cartesian):
has_point_id = ('POINT_ID' in param_values[0].columns and 'POINT_ID' in param_values[1].columns)

# Convert FITS big-endian columns to native byte order:
for df in param_values:
    for col in df.columns:
        arr = df[col].to_numpy()
        if arr.dtype.kind in 'biufc' and arr.dtype.byteorder == '>':
            df[col] = arr.astype(arr.dtype.newbyteorder('='), copy=True)
print("Parameter DataFrames converted to native byte order.")

# Collect all physical parameters:
all_param_names = []
for names in param_names:
    for name in names:
        if name.lower() not in excluded_params:
            if name not in all_param_names:
                all_param_names.append(name)

# Helper functions:
def format_value(value):
    if isinstance(value, (float, np.floating)):
        return f'{value:.3g}'
    return str(value)

def get_parameter_unit(component, parameter):
    """Retrieve the FITS unit corresponding to a parameter."""
    names = list(param_names[component])
    if parameter not in names:
        return ''
    unit = param_units[component][names.index(parameter)]
    return (str(unit) if unit is not None and str(unit).strip() else '')

# Create parameter controls:
sliders = {}
for p in all_param_names:
    # Collect values from both components.
    unique_vals = []
    for df in param_values:
        if p in df.columns:
            unique_vals.extend(df[p].dropna().unique().tolist())
    unique_vals = sorted(set(unique_vals))
    if not unique_vals:
        continue

    # Numerical parameter:
    if all(isinstance(v, (int, float, np.number)) and not isinstance(v, (bool, np.bool_)) for v in unique_vals):
        min_v = float(min(unique_vals))
        max_v = float(max(unique_vals))
        # Slider resolution:
        step_v = max((max_v - min_v) / 200, 1e-6)
        # Initialize at an actual simulated value:
        initial_v = float(unique_vals[len(unique_vals) // 2])
        if min_v == max_v:
            sliders[p] = widgets.FloatSlider(
                value=min_v,
                min=min_v,
                max=min_v + 1e-5,
                step=1e-6,
                description=p,
                disabled=True,
                readout_format='.4g',
                continuous_update=False,
                style={'description_width': 'initial'},
                layout={'width': '450px'}
            )
        else:
            sliders[p] = widgets.FloatSlider(
                value=initial_v,
                min=min_v,
                max=max_v,
                step=step_v,
                description=p,
                readout_format='.4g',
                continuous_update=False,
                style={'description_width': 'initial'},
                layout={'width': '450px'}
            )

    # Categorical parameter:
    else:
        sliders[p] = widgets.Dropdown(
            options=unique_vals,
            value=unique_vals[0],
            description=p,
            style={'description_width': 'initial'},
            layout={'width': '450px'}
        )

# Nearest-neighbour matching:
def find_nearest_spectrum(df, selected_params):
    """
    Find the closest available spectrum using normalized Euclidean distance in parameter space.
    """
    dist = np.zeros(len(df), dtype=float)
    for name, target_val in selected_params.items():
        if name not in df.columns:
            continue
        col = df[name]
        if pd.api.types.is_numeric_dtype(col):
            col_values = col.to_numpy(dtype=float)
            # Normalize by the standard deviation:
            scale = col.std()
            if not np.isfinite(scale) or scale == 0:
                scale = 1.0
            contribution = ((col_values - target_val) / scale) ** 2
            contribution[~np.isfinite(contribution)] = np.inf
            dist += contribution
        else:
            # Categorical parameters must match exactly:
            dist += np.where(col.to_numpy() == target_val, 0.0, 1e6)
    if not np.any(np.isfinite(dist)):
        raise ValueError("No valid spectrum found for the selected parameters.")
    best_idx = int(np.argmin(dist))
    return best_idx, dist[best_idx]

# Plotting function:
def plot_selected_spectrum(**selected_params):
    fig, axs = plt.subplots(figsize=(15, 6), ncols=2, nrows=1)
    fig.suptitle('Explore the XSTAR spectral grid', fontsize=13)
    for j in range(2):
        df = param_values[j]
        # Find the nearest available simulation.
        best_idx, best_distance = find_nearest_spectrum(df, selected_params)
        row = df.iloc[best_idx]
        # Build parameter description:
        title_items = []
        for name in param_names[j]:
            if name.lower() in excluded_params:
                continue
            if name not in row.index:
                continue
            val = row[name]
            val_str = format_value(val)
            unit = get_parameter_unit(j, name)
            if unit:
                title_items.append(f'{name} = {val_str} ({unit})')
            else:
                title_items.append(f'{name} = {val_str}')
        param_str = ' | '.join(title_items)
        # Simulation ID:
        if has_point_id and 'POINT_ID' in row.index:
            id_str = f'POINT_ID: {row["POINT_ID"]}'
        else:
            id_str = f'Model Index: {best_idx}'
        # Plot spectrum:
        axs[j].plot(energy_range, spectra_range[j][best_idx])
        axs[j].set_title(f'{labels[j]} | {id_str}\n {param_str}',fontsize=8,pad=12)
        axs[j].set_xlabel('Energy (keV)')
        if j == 0:
            axs[j].set_yscale('log')
            axs[j].set_ylabel(r'Flux ($\mathrm{ph\ s^{-1}\ cm^{-2}}$)')
        else:
            axs[j].set_ylabel('Transmitted fraction')
    plt.tight_layout()
    plt.show()

# Interactive interface:
slider_list = list(sliders.values())
rows = [widgets.HBox(slider_list[i:i+2]) for i in range(0, len(slider_list), 2)]
controls = widgets.VBox(rows)
# Connect sliders to plotting function:
output = widgets.interactive_output(plot_selected_spectrum, sliders)
# Display interface.
display(widgets.HTML(
        '<h3>Explore the XSTAR spectral grid</h3>'
        '<p>Adjust the physical parameters to find the '
        'nearest available XSTAR simulation.</p>'
    ),
    controls,
    output
)